In [1]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [2]:
torch.cuda.device_count()

1

In [3]:
def try_gpu(i=0):  #@save
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():  #@save
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}')
             for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

In [4]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

In [5]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [6]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y

tensor([[0.0635, 0.0695, 0.6066],
        [0.2561, 0.2634, 0.9914]])

In [9]:
Z = X.cuda(0)
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')


In [10]:
Y + Z

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [11]:
X+Z

tensor([[2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')

In [13]:
Z.cuda(0) is Z

True

In [14]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

In [15]:
net(X)

tensor([[-0.5971],
        [-0.5971]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [16]:
net[0].weight.data.device

device(type='cuda', index=0)

In [21]:
X1 = torch.rand(10000, 20000, device=try_gpu())
Y1 = torch.rand(20000, 10000, device=try_gpu())

Z1 = torch.matmul(X1, Y1)

In [22]:
X2 = torch.rand(10000, 20000)
Y2 = torch.rand(20000, 10000)
Z2 = torch.matmul(X2, Y2)

In [23]:
import time
import torch

def benchmark(fn, device, repeats=10, warmup=3):
    # 预热，避免把CUDA初始化等时间算进去
    for _ in range(warmup):
        fn()

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    start = time.perf_counter()

    for _ in range(repeats):
        fn()

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    return (time.perf_counter() - start) / repeats

In [24]:
cpu = torch.device("cpu")
gpu = torch.device("cuda:0")

n = 2048

A_cpu = torch.randn(n, n)
B_cpu = torch.randn(n, n)

# 提前传输，这里只比较计算速度
A_gpu = A_cpu.to(gpu)
B_gpu = B_cpu.to(gpu)

cpu_time = benchmark(
    lambda: A_cpu @ B_cpu,
    cpu,
    repeats=5
)

gpu_time = benchmark(
    lambda: A_gpu @ B_gpu,
    gpu,
    repeats=5
)

print("CPU平均时间：", cpu_time)
print("GPU平均时间：", gpu_time)
print("加速倍数：", cpu_time / gpu_time)

CPU平均时间： 0.03445119999814779
GPU平均时间： 0.0017107200052123516
加速倍数： 20.13842118709038


In [25]:
n = 32

A_cpu = torch.randn(n, n)
B_cpu = torch.randn(n, n)

A_gpu = A_cpu.to(gpu)
B_gpu = B_cpu.to(gpu)

cpu_time = benchmark(
    lambda: A_cpu @ B_cpu,
    cpu,
    repeats=5000
)

gpu_time = benchmark(
    lambda: A_gpu @ B_gpu,
    gpu,
    repeats=5000
)

print("CPU平均时间：", cpu_time)
print("GPU平均时间：", gpu_time)

CPU平均时间： 1.7549299995880575e-05
GPU平均时间： 9.114755999762565e-05


In [26]:
device = torch.device("cuda:0")

net = net.to(device)
for name, param in net.named_parameters():
    print(name, param.shape, param.device)

0.weight torch.Size([1, 3]) cuda:0
0.bias torch.Size([1]) cuda:0


In [27]:
weight_gpu = net[0].weight
print(weight_gpu)
print(weight_gpu.device)

Parameter containing:
tensor([[ 0.0829, -0.4663,  0.2827]], device='cuda:0', requires_grad=True)
cuda:0


In [28]:
device = torch.device("cuda:0")

n = 512
num_operations = 1000

A = torch.randn(n, n, device=device)
B = torch.randn(n, n, device=device)

# 预热
for _ in range(5):
    C = A @ B

torch.cuda.synchronize(device)
start = time.perf_counter()

norms_cpu = []

for _ in range(num_operations):
    C = A @ B
    norm_value = torch.linalg.matrix_norm(C, ord="fro")

    # item()把GPU上的标量传回CPU
    norms_cpu.append(norm_value.item())

torch.cuda.synchronize(device)
time_one_by_one = time.perf_counter() - start

print("逐个传输时间：", time_one_by_one)

逐个传输时间： 0.5159936000127345


In [29]:
logs_gpu = torch.empty(
    num_operations,
    device=device
)

torch.cuda.synchronize(device)
start = time.perf_counter()

for i in range(num_operations):
    C = A @ B
    logs_gpu[i] = torch.linalg.matrix_norm(C, ord="fro")

# 只传输一次
norms_cpu = logs_gpu.cpu()

torch.cuda.synchronize(device)
time_transfer_once = time.perf_counter() - start

print("最后统一传输时间：", time_transfer_once)
print("速度比：", time_one_by_one / time_transfer_once)

最后统一传输时间： 0.19590160000370815
速度比： 2.6339427549492576
